# PaviaU 1D CNN

Pixel-wise hyperspectral image classification using a one-dimensional convolutional neural network.

This notebook is a clean, reproducible version of the supplied experiment. The experiment uses a class-wise 20% training split, `/255` normalization, a two-block 1D CNN, and validation-based evaluation.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

In [ ]:
import random
import numpy as np
import tensorflow as tf

from src.config import *
from src.data import load_pavia, make_split
from src.model import build_model
from src.evaluate import evaluate_predictions, save_training_history, save_classification_map

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

## 1. Load PaviaU dataset

Place `PaviaU.mat` and `PaviaU_gt.mat` inside `data/`.

In [ ]:
image, ground_truth = load_pavia(PAVIA_IMAGE, PAVIA_GT)
print("Image shape:", image.shape)
print("Ground-truth shape:", ground_truth.shape)

## 2. Prepare training and validation samples

In [ ]:
x_train, y_train, x_val, y_val, x_all, n_classes, scene_shape = make_split(
    image, ground_truth, TRAIN_FRACTION, SEED
)
print("Training samples:", len(x_train))
print("Validation samples:", len(x_val))
print("Number of classes:", n_classes)
print("Input shape:", x_train.shape[1:])

## 3. Build the 1D CNN

In [ ]:
model = build_model(x_train.shape[1], n_classes, DROPOUT, LEARNING_RATE)
model.summary()

## 4. Train

In [ ]:
history = model.fit(
    x_train, y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    verbose=1,
    validation_data=(x_val, y_val),
)

In [ ]:
save_training_history(history, FIGURES_DIR)

## 5. Generate the classification map

In [ ]:
probabilities = model.predict(x_all, batch_size=1024, verbose=1)
predicted_labels = np.argmax(probabilities, axis=1) + 1
pred_map = predicted_labels.reshape(scene_shape)
print("Prediction map shape:", pred_map.shape)
save_classification_map(pred_map, FIGURES_DIR)

## 6. Evaluate labeled pixels

In [ ]:
metrics = evaluate_predictions(ground_truth, pred_map, METRICS_DIR, FIGURES_DIR)
print(f"Overall Accuracy: {metrics['overall_accuracy'] * 100:.2f}%")
print(f"Cohen's Kappa: {metrics['kappa']:.4f}")
print("Per-class accuracy:")
for cls, value in metrics["per_class_accuracy"].items():
    print(f"  Class {cls}: {value * 100:.2f}%")

## 7. Save model and prediction

Generated files are written to `models/` and `results/`.

In [ ]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)
model.save(MODEL_DIR / "paviaU_1d_cnn.keras")
from scipy.io import savemat
savemat(RESULTS_DIR / "predicted_map.mat", {"outData": pred_map})
print("Saved model and prediction map.")